# Phase 01 — Scraping Berita 4 Bank (BCA, BRI, BNI, Mandiri) dari CNBC Indonesia

Notebook **mandiri (self-contained)**. Mengumpulkan berita ekonomi 4 bank besar dari CNBC Indonesia, mengikuti pendekatan tutorial scraping (`requests` + `BeautifulSoup` + `csv`).

### Cara pakai
- **Nilai penentu ada LANGSUNG di dalam fungsi** (bukan blok konfigurasi terpisah), ditandai komentar `# <- bisa diubah` di sampingnya. Cari komentar itu untuk mengganti keyword, tanggal, jeda, dsb.
- Sumber: halaman pencarian CNBC — `https://www.cnbcindonesia.com/search?query=<kw>&page=<N>&fromdate=YYYY/MM/DD&todate=YYYY/MM/DD`.
- Filter 4 bank lewat **keyword** (nama + kode saham), hasil **digabung & dedupe** via `id_artikel`.
- **Periode:** 20 Oktober 2024 – 20 Oktober 2026 (batas waktu untuk membatasi volume data; tanpa klaim kausalitas).
- **Checkpoint/resume:** hasil ditulis *append* ke CSV; `id_artikel` yang sudah ada dilewati → aman bila terputus.
- Isi disimpan mentah (cleaning & pemecahan kalimat di Phase 02).

> Catatan etika: data untuk riset akademik non-komersial; patuhi Ketentuan Penggunaan CNBC Indonesia.

## 1. Import

In [1]:
import re
import csv
import time
from pathlib import Path
from datetime import datetime

import requests
from bs4 import BeautifulSoup
import pandas as pd

## 2. Utilitas jaringan & ekstraksi isi artikel

`fetch()` mengunduh dengan retry. `ambil_isi_artikel()` membuka halaman artikel dan mengambil isi dari `div.detail-text` (gabung paragraf `<p>`), sama seperti tutorial.

In [2]:
def fetch(url):
    headers = {'User-Agent': 'Chrome/120.0.0.0'}

    # Bagian untuk memvalidasi jika link gagal diambil
    for percobaan in range(1, 4):
        try:
            r = requests.get(url, headers=headers, timeout=30)
            if r.status_code == 200:
                return r
            if r.status_code in (404, 410):
                return None
        except requests.RequestException as e:
            print(f"    [ERR] {percobaan}/3: {type(e).__name__}")
        time.sleep(2 ** percobaan)
    return None

def ambil_isi_artikel(link):
    r = fetch(link)
    if r is None:
        return ""
    soup = BeautifulSoup(r.text, "lxml")
    div = soup.find("div", class_="detail-text") # Class Konten berita 
    if div is None:
        return ""
    
    # buang elemen non-teks + blok video sisipan
    for buang in div.find_all(["script", "style", "iframe"]):
        buang.decompose()
    for v in div.find_all("a", class_=re.compile(r"autovideocnbc")):
        v.decompose()
    paragraf = [p.get_text(" ", strip=True) for p in div.find_all("p")]
    teks = " ".join(p for p in paragraf if p)
    teks = re.sub(r"\[Gambas:[^\]]*\]", " ", teks)
    teks = re.sub(r"\(\s*[a-z]{2,4}\s*/\s*[a-z]{2,4}\s*\)\s*$", "", teks).strip()
    teks = re.sub(r"\s+", " ", teks).strip()
    return teks

## 3. Baca halaman hasil pencarian

`cari_last_page()` menentukan jumlah halaman (dari angka paginasi, seperti tutorial). `parse_satu_halaman()` mengambil judul/link/waktu tiap artikel teks pada satu halaman hasil pencarian (membuang video).

In [5]:
def search_url(keyword, page, fromdate, todate):
    kw = keyword.replace(" ", "+")
    return f"https://www.cnbcindonesia.com/search?query={kw}&page={page}&fromdate={fromdate}&todate={todate}"


def cari_last_page(keyword, fromdate, todate):
    r = fetch(search_url(keyword, 1, fromdate, todate))
    if r is None:
        return 1
    soup = BeautifulSoup(r.text, "lxml")
    angka = []
    for a in soup.find_all("a"):
        try:
            angka.append(int(a.text.strip()))
        except ValueError:
            continue
    last = max(angka) if angka else 1
    return min(last, 200) # max paginasi sebanyak 200

# Logic crawl untuk mengambil semua konten relevan dari search url
def parse_satu_halaman(html):
    pola_url = re.compile(r"/(\w+)/(\d{14})-(\d+)-(\d+)/")
    soup = BeautifulSoup(html, "lxml")
    hasil = []
    for art in soup.find_all("article"):

        # validasi per step untuk tidak mengambil data tidak penting seperti link iklan
        a = art.find("a", href=True)
        if not a:
            continue
        
        link = a["href"].split("#")[0]
        m = pola_url.search(link)
        if not m:
            continue

        kanal, ts, kode_tipe, id_artikel = m.groups()
        if kode_tipe in ("18", "19"): # tipe video
            continue

        h2 = art.find("h2")
        if h2 and h2.get_text(strip=True):
            judul = h2.get_text(strip=True)
        else:
            img = a.find("img")
            judul = (img.get("alt").strip() if img and img.get("alt") else a.get_text(" ", strip=True))
        spans = a.find_all("span")
        waktu = spans[-1].get_text(strip=True) if spans else ""
        hasil.append({"id_artikel": id_artikel, "kanal": kanal, "url": link, "timestamp": ts, "judul": judul, "waktu": waktu})
    return hasil

## 4. Jalankan scraping (dengan checkpoint & resume)

Semua nilai penentu (keyword 4 bank, rentang tanggal, mode pilot, jeda, lokasi output) ada **langsung di dalam fungsi `jalankan()`** dan ditandai `# <- bisa diubah`. Tiap artikel ditulis *append* ke CSV; `id_artikel` yang sudah ada dilewati.

In [9]:
def jalankan(keywords, fromdate, todate, maks_halaman, jeda, output_csv):
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    header = ["id_artikel", "keyword", "kanal", "url", "timestamp",
              "judul", "waktu_listing", "isi_mentah", "isi_panjang", "tanggal_scrape"]

    # safeguard personal agar tidak mengulang scrawl dan scrape
    sudah = set()
    if output_csv.exists():
        with open(output_csv, "r", encoding="utf-8", newline="") as f:
            for row in csv.DictReader(f):
                sudah.add(str(row.get("id_artikel", "")))
    # print(f"Mode {'PILOT' if PILOT else 'FULL'} | periode {fromdate}..{todate} | keyword {keywords}")
    # print(f"[RESUME] {len(sudah)} artikel sudah ada, akan dilewati.\n")

    file_baru = not output_csv.exists()
    f = open(output_csv, "a", encoding="utf-8", newline="", buffering=1)  # line-buffered (aman bila terputus)
    writer = csv.DictWriter(f, fieldnames=header)
    if file_baru:
        writer.writeheader()

    n_baru = 0

    for keyword in keywords:
        last_page = min(cari_last_page(keyword, fromdate, todate), maks_halaman)
        print(f"=== keyword '{keyword}': {last_page} halaman ===")
        time.sleep(jeda)
        for page in range(1, last_page + 1):
            r = fetch(search_url(keyword, page, fromdate, todate))
            time.sleep(jeda)
            if r is None:
                print(f"  [hal {page}] gagal, lanjut")
                continue
            for a in parse_satu_halaman(r.text):
                if a["id_artikel"] in sudah:
                    continue   # sudah pernah (keyword lain / run sebelumnya) -> dedupe
                isi = ambil_isi_artikel(a["url"])
                time.sleep(jeda)
                writer.writerow({
                    "id_artikel": a["id_artikel"], "keyword": keyword, "kanal": a["kanal"],
                    "url": a["url"], "timestamp": a["timestamp"], "judul": a["judul"],
                    "waktu_listing": a["waktu"], "isi_mentah": isi, "isi_panjang": len(isi),
                    "tanggal_scrape": datetime.now().isoformat(timespec="seconds"),
                })
                sudah.add(a["id_artikel"])
                n_baru += 1
                print(f"  [{keyword} hal {page}] +{a['judul'][:60]}")

    f.close()
    print()
    print(f"selesai, {n_baru} artikel baru ditambahkan sesi ini -> {output_csv}")
    
    return output_csv

In [10]:
output_csv = jalankan(
    keywords= ['BCA'],
    # keywords = [
    # "BCA", "BBCA", "BRI", "BBRI",
    # "BNI", "BBNI", "Mandiri", "BMRI"
    # ],
    fromdate = "2024/10/20",  # periode awal
    todate   = "2024/10/25",  # periode akhir
    maks_halaman = 200,
    jeda = 1.0, # lag antar scrapping
    output_csv = "data/raw/test.csv"   # <- bisa diubah: lokasi output
)

AttributeError: 'str' object has no attribute 'parent'

## 5. Verifikasi hasil

In [ ]:
df = pd.read_csv(output_csv, dtype={"id_artikel": str})
print(f"Total artikel : {len(df)}")
print(f"id unik       : {df['id_artikel'].nunique()} (duplikat: {len(df)-df['id_artikel'].nunique()})")
print(f"isi < 150 char: {(df['isi_panjang'] < 150).sum()}")
print(f"rata panjang  : {df['isi_panjang'].mean():.0f} char")
print("\nJumlah per keyword:")
print(df["keyword"].value_counts())
df[["id_artikel", "keyword", "timestamp", "isi_panjang", "judul"]].head(10)

In [ ]:
# Contoh isi satu artikel (cek kualitas ekstraksi)
c = df[df["isi_panjang"] >= 150].iloc[0]
print("JUDUL :", c["judul"])
print("URL   :", c["url"])
print("-" * 70)
print(c["isi_mentah"][:1200], "...")

## 6. Menjalankan full-run (4 bank, 20 Okt 2024 – 20 Okt 2026)

Di dalam fungsi `jalankan()` (sel bagian 4), ubah baris `PILOT = True` menjadi **`PILOT = False`**, lalu jalankan ulang sel tersebut. Semua keyword 4 bank akan di-crawl untuk periode penuh, digabung & dedupe via `id_artikel`. Aman ditinggal: bila terputus, jalankan ulang — artikel yang sudah ada di `data/raw/cnbc_4bank_full.csv` dilewati.

> Jika suatu saat `/search` berhenti mengembalikan `<article>` (situs pindah ke rendering JavaScript), gunakan endpoint JSON `GET https://www.cnbcindonesia.com/api/v2/search?query=<kw>&fromdate=..&todate=..&start=0&limit=100` yang isinya setara (field `data[].strjudul`, `data[].strisi`, `data[].dtnewsdate`, `data[].articleUrl`; paginasi via `start`/`limit`).